Перед запуском убедитесь, что в корне проекта есть файл .env и в нем заполнены выданные вам креды подключения к базам данных и хранилищу

In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine

/tmp/ipykernel_94929/816847089.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
# подгружаем .env
load_dotenv()

True

In [3]:
# Считываем все креды
src_host = os.environ.get('DB_SOURCE_HOST')
src_port = os.environ.get('DB_SOURCE_PORT')
src_username = os.environ.get('DB_SOURCE_USER')
src_password = os.environ.get('DB_SOURCE_PASSWORD')
src_db = os.environ.get('DB_SOURCE_NAME') 

dst_host = os.environ.get('DB_DESTINATION_HOST')
dst_port = os.environ.get('DB_DESTINATION_PORT')
dst_username = os.environ.get('DB_DESTINATION_USER')
dst_password = os.environ.get('DB_DESTINATION_PASSWORD')
dst_db = os.environ.get('DB_DESTINATION_NAME')

s3_bucket = os.environ.get('S3_BUCKET_NAME')
s3_access_key = os.environ.get('AWS_ACCESS_KEY_ID')
s3_secret_access_key = os.environ.get('AWS_SECRET_ACCESS_KEY')

In [6]:
import os
import boto3
from dotenv import load_dotenv
from botocore.exceptions import ClientError

class Config:
    load_dotenv()
    AWS_ACCESS_KEY_ID = os.environ.get('AWS_ACCESS_KEY_ID')
    AWS_SECRET_ACCESS_KEY = os.environ.get('AWS_SECRET_ACCESS_KEY')
    S3_SERVICE_NAME = 's3'
    S3_ENDPOINT_URL = 'https://storage.yandexcloud.net'
    BUCKET_NAME = os.environ.get('S3_BUCKET_NAME')


def get_session():
    session = boto3.session.Session()
    return session.client(
        service_name=Config.S3_SERVICE_NAME,
        endpoint_url=Config.S3_ENDPOINT_URL,
        aws_access_key_id=Config.AWS_ACCESS_KEY_ID,
        aws_secret_access_key=Config.AWS_SECRET_ACCESS_KEY
    )

def test_read_access(s3):
    try:
        response = s3.list_objects(Bucket=Config.BUCKET_NAME)
        if 'Contents' in response:
            print("Содержимое бакета:")
            for obj in response['Contents']:
                print(f" - {obj['Key']}")
        else:
            print("Бакет пустой")
        return True
    except ClientError as e:
        print(f"Ошибка доступа: {e}")
        return False

def test_write_access(s3):
    test_content = "test content"
    test_key = "test-file.txt"

    try:
        s3.put_object(
            Bucket=Config.BUCKET_NAME,
            Key=test_key,
            Body=test_content
        )
        print(f"Файл {test_key} успешно записан")
        return True
    except ClientError as e:
        print(f"Ошибка записи: {e}")
        return False

if __name__ == "__main__":
    s3 = get_session()

    print("Проверка чтения...")
    if test_read_access(s3):
        print("Чтение: OK")

    print("\nПроверка записи...")
    if test_write_access(s3):
        print("Запись: OK")

Проверка чтения...
Бакет пустой
Чтение: OK

Проверка записи...
Файл test-file.txt успешно записан
Запись: OK


In [9]:
import boto3

class Config:
    AWS_ACCESS_KEY_ID = s3_access_key
    AWS_SECRET_ACCESS_KEY = s3_secret_access_key
    S3_SERVICE_NAME = 's3'
    S3_ENDPOINT_URL = 'https://storage.yandexcloud.net'


def get_session():
    session = boto3.session.Session()
    return session.client(
        service_name=Config.S3_SERVICE_NAME,
        endpoint_url=Config.S3_ENDPOINT_URL,
        aws_access_key_id=Config.AWS_ACCESS_KEY_ID,
        aws_secret_access_key=Config.AWS_SECRET_ACCESS_KEY
    )


bucket_name = s3_bucket

s3 = get_session()

if s3.list_objects(Bucket=bucket_name).get('Contents'):
    for key in s3.list_objects(Bucket=bucket_name)['Contents']:
        print(key['Key'])

test-file.txt


In [ ]:
# Создадим соединения
src_conn = create_engine(f'postgresql://{src_username}:{src_password}@{src_host}:{src_port}/{src_db}')
dst_conn = create_engine(f'postgresql://{dst_username}:{dst_password}@{dst_host}:{dst_port}/{dst_db}')

In [ ]:
# Пример выгрузки данных из БД
TABLE = ''
SQL = f'select * from {TABLE}'
data = pd.read_sql(SQL, src_conn)

А дальше, творите!